In [2]:
import sys
sys.path.append('../src')

from datetime import date
from bank.bank import Bank, ProhibitedOperationTime
from bank.client import Client, InvalidPassword, Contact
from account.bank_account import BankAccount
from account.premium_account import PremiumAccount
from account.savings_account import SavingsAccount
from account.investment_account import InvestmentAccount, Asset
from account.enums import AssetType


# --------------------------------------
# 0️⃣ Хранилище паролей
# --------------------------------------

passwords = {
    "ivan": "1234",
    "maria": "abcd",
    "sergey": "qwerty",
}


# --------------------------------------
# 1️⃣ Создание банка и клиентов
# --------------------------------------

bank = Bank()

clients = [
    Client(
        name="Иван Иванов",
        client_id="c1",
        date_of_birth=date(1993, 2, 15),
        login="ivan",
        password=passwords["ivan"],
        contacts=[Contact("email", "ivan@mail.com")],
    ),
    Client(
        name="Мария Петрова",
        client_id="c2",
        date_of_birth=date(1998, 7, 20),
        login="maria",
        password=passwords["maria"],
        contacts=[Contact("phone", "+79991234567")]
    ),
    Client(
        name="Сергей Сидоров",
        client_id="c3",
        date_of_birth=date(1983, 11, 5),
        login="sergey",
        password=passwords["sergey"],
        contacts=[Contact("email", "sergey@mail.com")]
    ),
]

for c in clients:
    bank.add_client(c)

print("Клиенты добавлены:", [c.name for c in bank.clients])

# --------------------------------------
# 2️⃣ Открытие счетов разных типов
# --------------------------------------

ivan = clients[0]
maria = clients[1]
sergey = clients[2]

# Иван
ivan_acc1 = BankAccount(owner_name=ivan.name)
ivan_acc2 = SavingsAccount(
    owner_name=ivan.name,
    min_balance=100,
    monthly_interest=1
)

bank.open_account(ivan.client_id, ivan_acc1)
bank.open_account(ivan.client_id, ivan_acc2)

# Мария
maria_acc1 = PremiumAccount(
    owner_name=maria.name,
    fee=5,
    overdraft_limit=500
)

maria_acc2 = InvestmentAccount(
    owner_name=maria.name,
    assets=[
        Asset(asset_type=AssetType.STOCK, name="AAPL", price=150, yearly_interest=5),
        Asset(asset_type=AssetType.ETF, name="SP500", price=200, yearly_interest=4)
    ]
)

bank.open_account(maria.client_id, maria_acc1)
bank.open_account(maria.client_id, maria_acc2)

# Сергей
sergey_acc1 = BankAccount(owner_name=sergey.name)
bank.open_account(sergey.client_id, sergey_acc1)


# --------------------------------------
# Проверка
# --------------------------------------

print("Счета открыты:")

for c in clients:
    accs = bank.search_accounts(client_id=c.client_id)
    print(f"{c.name}: {[str(a) for a in accs]}")

# --------------------------------------
# 3️⃣ Аутентификация клиентов
# --------------------------------------
sessions = {}

# Правильные логины
for c in clients:
    try:
        sid = bank.authenticate_client(c.login, passwords[c.login])
        sessions[c.client_id] = sid
        print(f"{c.name} вошел в систему, session_id: {sid}")
    except InvalidPassword:
        print(f"Ошибка пароля для {c.name}")

# Неправильный пароль для теста блокировки
try:
    bank.authenticate_client("ivan", "wrong")
except InvalidPassword:
    print("Попытка неправильного пароля для Ивана, помечаем как подозрительное действие")

# --------------------------------------
# 4️⃣ Депозиты и снятия
# --------------------------------------
# Иван делает несколько операций
ivan_session_id = sessions["c1"]
ivan_accounts = bank.search_accounts(client_id="c1")

bank.deposit(ivan_session_id, ivan_accounts[0].id, 1000)
bank.deposit(ivan_session_id, ivan_accounts[1].id, 500)
bank.withdraw(ivan_session_id, ivan_accounts[0].id, 200)

# Мария с премиум и инвестиционным
maria_session_id = sessions["c2"]
maria_accounts = bank.search_accounts(client_id="c2")

bank.deposit(maria_session_id, maria_accounts[0].id, 3000)
bank.withdraw(maria_session_id, maria_accounts[0].id, 100)
print("Проектируемый доход по инвестициям Марии:", maria_accounts[1].project_yearly_growth())

# Сергей обычный банк
sergey_session_id = sessions["c3"]
sergey_account = bank.search_accounts(client_id="c3")[0]
bank.deposit(sergey_session_id, sergey_account.id, 700)
bank.withdraw(sergey_session_id, sergey_account.id, 50)

# --------------------------------------
# 6️⃣ Заморозка/разморозка
# --------------------------------------
bank.freeze_account(ivan_accounts[1].id)
print("Статус счета Ивана после заморозки:", ivan_accounts[1].status)
bank.unfreeze_account(ivan_accounts[1].id)
print("Статус счета Ивана после разблокировки:", ivan_accounts[1].status)

# --------------------------------------
# 7️⃣ Балансы и рейтинг клиентов
# --------------------------------------
for c in clients:
    print(f"{c.name} общий баланс:", bank.get_total_balance(c.client_id))

ranking = bank.get_clients_ranking()
print("Рейтинг клиентов по балансу:", [c.name for c in ranking])

# --------------------------------------
# 8️⃣ Сессии клиентов
# --------------------------------------
print("Сессии клиентов:")
for client_id, sid in sessions.items():
    print(f"{client_id}: {sid}")

# Добавим ещё сессии для проверки TOO_MANY_SESSIONS
for i in range(11):
    try:
        bank.authenticate_client("ivan", "1234")
    except InvalidPassword:
        pass

print("Много неправильных паролей (LOGIN_FAILED + блокировка клиента)")
try:
    bank.authenticate_client("ivan", "wrong1")
except InvalidPassword:
    pass

try:
    bank.authenticate_client("ivan", "wrong2")
except InvalidPassword:
    pass

try:
    bank.authenticate_client("ivan", "wrong3")
except InvalidPassword:
    pass

print("Попытка входа в замороженный аккаунт")
try:
    bank.authenticate_client("ivan", "1234")
except Exception as e:
    print("Login after freeze:", type(e).__name__)

print("Слишком большая операция (BIG_TRANSACTION)")

bank.deposit(ivan_session_id, ivan_accounts[0].id, 20000)

print("Слишком много сессий (TOO_MANY_SESSIONS)")

for _ in range(12):
    try:
        bank.authenticate_client("maria", "abcd")
    except InvalidPassword:
        pass

try:
    bank.withdraw(ivan_session_id, maria_accounts[0].id, 10)
except Exception as e:
    print("Access чужого счета:", type(e).__name__)

try:
    bank.withdraw(ivan_session_id, "fake_account_id", 10)
except Exception as e:
    print("Fake account:", type(e).__name__)

try:
    bank.deposit("fake_session", ivan_accounts[0].id, 10)
except Exception as e:
    print("Fake session:", type(e).__name__)

print("Ночная операция (NIGHT_OPERATION)")

def fake_night():
    raise ProhibitedOperationTime

bank._is_operations_allowed = fake_night

try:
    bank.withdraw(ivan_session_id, ivan_accounts[0].id, 10)
except ProhibitedOperationTime:
    pass

for c in clients:
    print(f"\n🚨 Fraud log for {c.name}")
    for fraud in c.frauds:
        print("-", fraud.fraud_type, fraud.created_at)


Клиенты добавлены: ['Иван Иванов', 'Мария Петрова', 'Сергей Сидоров']
Счета открыты:
Иван Иванов: ['BankAccount | Owner: Иван Иванов | Account: ****d430 | Status: active | Balance: 0 RUB', 'SavingsAccount | Owner: Иван Иванов | Account: ****3786 | Status: active | Balance: 0 RUB |Min balance: 100 | Monthly interest: 1%']
Мария Петрова: ['PremiumAccount | Owner: Мария Петрова | Account: ****9845 | Status: active | Balance: 0 RUB |Fee: 5 | Overdraft limit: 500', 'InvestmentAccount | Owner: Мария Петрова | Account: ****3348 | Status: active | Balance: 0 RUB |Assets: [AAPL(AssetType.STOCK, price=150, interest=5%), SP500(AssetType.ETF, price=200, interest=4%)]']
Сергей Сидоров: ['BankAccount | Owner: Сергей Сидоров | Account: ****1beb | Status: active | Balance: 0 RUB']
Иван Иванов вошел в систему, session_id: a70d4b7e-a76a-4ff4-bc6b-bd76fe6c5c7c
Мария Петрова вошел в систему, session_id: 316ab1c7-8cac-476a-8a25-918c9f24d1ab
Сергей Сидоров вошел в систему, session_id: b47ab547-82cc-4fd1-818